In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import sys

sys.path.insert(
    0,
    "/content/drive/MyDrive/VulnSight"
)

In [4]:
import pandas as pd
import numpy as np
import csv
import pickle
import re
import torch
import sklearn
import os
import random
import clang
from clang import *
from clang import cindex
from pathlib import Path
from tokenizers import ByteLevelBPETokenizer
from tokenizers.implementations import ByteLevelBPETokenizer
from tokenizers.processors import BertProcessing
from torch.utils.data import Dataset, DataLoader, IterableDataset
from transformers import RobertaConfig
from transformers import RobertaForMaskedLM, RobertaForSequenceClassification
from transformers import RobertaTokenizerFast
from transformers import DataCollatorForLanguageModeling
from transformers import Trainer, TrainingArguments
from transformers.modeling_outputs import SequenceClassifierOutput

from tokenizers import Tokenizer
from tokenizers import normalizers,decoders
from tokenizers.normalizers import StripAccents, unicode_normalizer_from_str, Replace
from tokenizers.processors import TemplateProcessing
from tokenizers import processors,pre_tokenizers
from tokenizers.models import BPE

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [6]:
## Deterministic/reproducible flags

seedlist = [42, 834, 692, 489, 901, 408, 819, 808, 531, 166]

seed = seedlist[0]
os.environ['PYTHONHASHSEED'] = str(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
random.seed(seed)
torch.backends.cudnn.enabled = True
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [7]:
os.environ['WANDB_DISABLED'] = 'true'

PROJECT_ROOT = Path("/content/drive/MyDrive/VulnSight")

In [8]:
## Tokenizer

from tokenizers.pre_tokenizers import PreTokenizer
from tokenizers.pre_tokenizers import Whitespace
from tokenizers import NormalizedString,PreTokenizedString
from typing import List

class MyTokenizer:

    cidx = cindex.Index.create()


    def clang_split(self, i: int, normalized_string: NormalizedString) -> List[NormalizedString]:
        ## Tokenize using clang
        tok = []
        tu = self.cidx.parse('tmp.c',
                       args=[''],
                       unsaved_files=[('tmp.c', str(normalized_string.original))],
                       options=0)
        for t in tu.get_tokens(extent=tu.cursor.extent):
            spelling = t.spelling.strip()

            if spelling == '':
                continue

            tok.append(NormalizedString(spelling))

        return(tok)

    def pre_tokenize(self, pretok: PreTokenizedString):
        pretok.split(self.clang_split)

## Load pre-trained tokenizers
vocab_path = str(PROJECT_ROOT / "tokenizer" / "drapgh-vocab.json")
merge_path = str(PROJECT_ROOT / "tokenizer" / "drapgh-merges.txt")

vocab, merges = BPE.read_file(vocab=vocab_path, merges=merge_path)

my_tokenizer = Tokenizer(BPE(vocab, merges, unk_token="<unk>"))

my_tokenizer.normalizer = normalizers.Sequence([StripAccents(), Replace(" ", "Ä")])
my_tokenizer.pre_tokenizer = PreTokenizer.custom(MyTokenizer())
my_tokenizer.post_processor = TemplateProcessing(
    single="<s> $A </s>",
    special_tokens=[
    ("<s>",0),
    ("<pad>",1),
    ("</s>",2),
    ("<unk>",3),
    ("<mask>",4)
    ]
)


In [9]:
mydataset = 'reveal'

In [10]:
my_tokenizer.enable_truncation(max_length=1024)
my_tokenizer.enable_padding(direction='right', pad_id=1, pad_type_id=0, pad_token='<pad>', length=None, pad_to_multiple_of=None)

In [11]:
def cleaner(code):
    ## Remove code comments
    pat = re.compile(r'(/\*([^*]|(\*+[^*/]))*\*+/)|(//.*)')
    code = re.sub(pat,'',code)
    code = re.sub('\n','',code)
    code = re.sub('\t','',code)
    return(code)

def process_encodings(encodings):
    input_ids=[]
    attention_mask=[]
    for enc in encodings:
        input_ids.append(enc.ids)
        attention_mask.append(enc.attention_mask)
    return {'input_ids':input_ids, 'attention_mask':attention_mask}

def replace_colname(x):
    x = x.rename(columns={
        'functionSource': 'func',
        'code': 'func',
        'label': 'target'
    })
    return x

In [12]:
if mydataset == 'devign':
    devign_dir = PROJECT_ROOT / "finetune" / "devign"
    train_index_path = devign_dir / "train.txt"
    valid_index_path = devign_dir / "valid.txt"
    devign_json_path = devign_dir / "Devign.json"

    for p in (train_index_path, valid_index_path, devign_json_path):
        if not p.exists():
            raise FileNotFoundError(f"Required Devign file not found: {p}")

    train_index = set()
    valid_index = set()

    with open(train_index_path) as f:
        for line in f:
            line = line.strip()
            if line:
                train_index.add(int(line))

    with open(valid_index_path) as f:
        for line in f:
            line = line.strip()
            if line:
                valid_index.add(int(line))

    mydata = pd.read_json(devign_json_path)
    m1 = mydata.iloc[list(train_index)].copy()
    m2 = mydata.iloc[list(valid_index)].copy()
    del mydata

elif mydataset == 'd2a':
    task = 'function'
    d2a_dir = PROJECT_ROOT / "finetune" / "d2a" / task
    train_path = d2a_dir / f"d2a_lbv1_{task}_train.csv"
    val_path = d2a_dir / f"d2a_lbv1_{task}_dev.csv"

    for p in (train_path, val_path):
        if not p.exists():
            raise FileNotFoundError(
                f"Required D2A file not found: {p}\n"
                "Update d2a_dir/train_path/val_path to match your Google Drive folders."
            )

    m1 = pd.read_csv(train_path)
    m2 = pd.read_csv(val_path)
    m1 = replace_colname(m1)
    m2 = replace_colname(m2)

else:
    dataset_dir = PROJECT_ROOT / "finetune" / mydataset.lower()

    train_path = dataset_dir / f"{mydataset.lower()}_train.pkl"
    val_path = dataset_dir / f"{mydataset.lower()}_val.pkl"

    for p in (train_path, val_path):
        if not p.exists():
            raise FileNotFoundError(
                f"Required dataset file not found: {p}\n"
                "Update train_path and val_path to match your Google Drive folders."
            )

    m1 = pd.read_pickle(train_path)
    m2 = pd.read_pickle(val_path)
    m1 = replace_colname(m1)
    m2 = replace_colname(m2)

In [13]:
class MyCustomDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
        n = len(self.labels)
        assert len(self.encodings["input_ids"]) == n
        assert len(self.encodings["attention_mask"]) == n

    def __getitem__(self, idx):
        item = {
            key: torch.tensor(values[idx], dtype=torch.long)
            for key, values in self.encodings.items()
        }
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)


def get_code_column(df):
    for col in ("func", "code", "functionSource"):
        if col in df.columns:
            return col
    raise ValueError(f"No code column found. Available columns: {list(df.columns)}")


def get_label_column(df):
    for col in ("target", "label", "combine"):
        if col in df.columns:
            return col
    raise ValueError(f"No label column found. Available columns: {list(df.columns)}")


MAX_LENGTH = 512
pad_id = my_tokenizer.token_to_id("<pad>")
if pad_id is None:
    raise ValueError("Tokenizer vocabulary does not contain the <pad> token.")

# Keep the tokenizer's own truncation setting consistent with the model input length.
my_tokenizer.enable_truncation(max_length=MAX_LENGTH)


def prepare_dataset(df, split_name):
    code_col = get_code_column(df)
    label_col = get_label_column(df)

    # Drop rows with missing labels; replace missing source code with an empty string.
    clean_df = df.loc[df[label_col].notna(), [code_col, label_col]].copy()
    code = clean_df[code_col].fillna("").astype(str).apply(cleaner).tolist()

    raw_labels = pd.to_numeric(clean_df[label_col], errors="raise")
    unique_labels = set(raw_labels.unique().tolist())
    if not unique_labels.issubset({0, 1, 0.0, 1.0}):
        raise ValueError(
            f"{split_name} labels must be binary 0/1; found {sorted(unique_labels)}"
        )
    labels = raw_labels.astype(int).tolist()

    encodings = my_tokenizer.encode_batch(code)
    input_ids, attention_masks = [], []

    for enc in encodings:
        ids = enc.ids[:MAX_LENGTH]
        mask = enc.attention_mask[:MAX_LENGTH]
        pad_length = MAX_LENGTH - len(ids)
        input_ids.append(ids + [pad_id] * pad_length)
        attention_masks.append(mask + [0] * pad_length)

    encoded_data = {
        "input_ids": input_ids,
        "attention_mask": attention_masks,
    }
    dataset = MyCustomDataset(encoded_data, labels)
    return dataset, labels


train_dataset, train_labels = prepare_dataset(m1, "Training")
val_dataset, val_labels = prepare_dataset(m2, "Validation")

if not train_labels or not val_labels:
    raise ValueError("Training and validation datasets must both contain at least one labeled row.")
if set(train_labels) != {0, 1}:
    raise ValueError(f"Training data must contain both classes 0 and 1; found {sorted(set(train_labels))}")

print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Input sequence length:", MAX_LENGTH)
print("Training label distribution:\n", pd.Series(train_labels).value_counts().sort_index())
print("Validation label distribution:\n", pd.Series(val_labels).value_counts().sort_index())

Training samples: 18187
Validation samples: 2273
Input sequence length: 512
Training label distribution:
 0    16387
1     1800
Name: count, dtype: int64
Validation label distribution:
 0    2063
1     210
Name: count, dtype: int64


In [14]:
## Pre-trained RoBERTa

MODEL_PATH = PROJECT_ROOT / "models" / "VulSight"
if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Pretrained model directory not found: {MODEL_PATH}")

model = RobertaForSequenceClassification.from_pretrained(str(MODEL_PATH))
print("Model parameters:", model.num_parameters())

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: /content/drive/MyDrive/VulnSight/models/VulSight
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.decoder.bias       | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.decoder.weight     | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model parameters: 124836866


### Custom loss function with class weights

In [15]:
cw = sklearn.utils.class_weight.compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=np.asarray(train_labels, dtype=int),
)

c_weights = torch.tensor(cw, dtype=torch.float32, device=device)
print("Class weights [class 0, class 1]:", c_weights.detach().cpu().tolist())

Class weights [class 0, class 1]: [0.554921567440033, 5.051944255828857]


In [16]:
criterion = torch.nn.CrossEntropyLoss(weight=c_weights)

class MyTrainer(Trainer):
    """Trainer that reports mean training loss and validation loss at each epoch."""

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._epoch_train_loss_sum = 0.0
        self._epoch_train_examples = 0
        self.best_val_loss = float("inf")
        self.best_epoch = None

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss = criterion(logits, labels)

        # Accumulate the unscaled batch loss only during training, not evaluation.
        if model.training:
            batch_size = int(labels.shape[0])
            self._epoch_train_loss_sum += float(loss.detach().float().item()) * batch_size
            self._epoch_train_examples += batch_size

        return (loss, outputs) if return_outputs else loss

    def evaluate(self, *args, **kwargs):
        metrics = super().evaluate(*args, **kwargs)
        val_loss = metrics.get("eval_loss")

        # Trainer calls evaluate at the end of each epoch with evaluation_strategy='epoch'.
        if val_loss is not None and self._epoch_train_examples > 0:
            epoch_train_loss = self._epoch_train_loss_sum / self._epoch_train_examples
            epoch_num = int(round(float(self.state.epoch or 0)))
            print(
                f"\nEpoch {epoch_num}: "
                f"train_loss={epoch_train_loss:.6f} | val_loss={float(val_loss):.6f}"
            )

            if float(val_loss) < self.best_val_loss:
                self.best_val_loss = float(val_loss)
                self.best_epoch = epoch_num
                print(f"New best checkpoint candidate: epoch {self.best_epoch}, val_loss={self.best_val_loss:.6f}")

            self._epoch_train_loss_sum = 0.0
            self._epoch_train_examples = 0

        return metrics

In [20]:
training_args = TrainingArguments(
    output_dir=str(PROJECT_ROOT / "models" / f"VulSight-MLP_{mydataset}"),
    per_device_train_batch_size=8,
    num_train_epochs=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,
    seed=seed,
    learning_rate=3e-5,
    fp16=torch.cuda.is_available(),
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_strategy="steps",
    logging_steps=100,
)

In [21]:
trainer = MyTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

In [22]:
train_result = trainer.train()

print("\n========== Training Summary ==========")
if trainer.best_epoch is not None:
    print(f"Best epoch (lowest validation loss): {trainer.best_epoch}")
    print(f"Best validation loss: {trainer.best_val_loss:.6f}")
else:
    print("Best epoch could not be determined; check whether epoch evaluation ran successfully.")

print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Final training metrics:", train_result.metrics)

Epoch,Training Loss,Validation Loss
1,0.816543,1.238904
2,0.720978,0.972790
3,0.768441,0.906083
4,0.862059,1.054629
5,0.827983,0.771611
6,0.723805,0.711295
7,0.840885,0.809870



Epoch 1: train_loss=0.776619 | val_loss=1.238904
New best checkpoint candidate: epoch 1, val_loss=1.238904


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Epoch 2: train_loss=0.768076 | val_loss=0.972790
New best checkpoint candidate: epoch 2, val_loss=0.972790


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...


Epoch 9: train_loss=0.590454 | val_loss=1.102292


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Epoch 10: train_loss=0.508593 | val_loss=1.052530


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


========== Training Summary ==========
Best epoch (lowest validation loss): 6
Best validation loss: 0.711295
Best checkpoint: /content/drive/MyDrive/VulnSight/models/VulSight-MLP_reveal/checkpoint-13644
Final training metrics: {'train_runtime': 5379.4544, 'train_samples_per_second': 33.808, 'train_steps_per_second': 4.227, 'total_flos': 4.78520076383232e+16, 'train_loss': 0.6979450175923638, 'epoch': 10.0}
